Using julia and its fast Laplacian solver, we speed up the computation of the resistance score. Tends to work better for large and sparse graphs (for smaller or denser graphs, it might even be slower).

In [1]:
import networkx as nx
import numpy as np
from scipy.sparse import diags

n = 5000
p = 0.005
m = 2
#G = nx.erdos_renyi_graph(n=n, p=p)
G = nx.barabasi_albert_graph(n=n, m=m)
adj = nx.to_numpy_array(G)
lap = diags(adj.sum(axis=1).flatten()) - adj


In [2]:
from propnetscore.node_selection_fast import NodeSelectionTaskFast
from scipy.sparse import csc_matrix
from juliacall import Main as jl

probs = np.random.dirichlet(np.ones(adj.shape[0]))
true_node = np.random.randint(adj.shape[0])
task = NodeSelectionTaskFast(adj, probs, true_node)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [3]:
import time
task1 = NodeSelectionTaskFast(adj, probs, true_node)

start_time = time.time()
L_inv = np.linalg.pinv(lap)
print("Laplacian inverse in Python computed in %.3f seconds" % (time.time() - start_time))
L_inv_jl = task1._julia_approx_laplacian_inverse(csc_matrix(adj))
#L_inv_jl @ lap
#L_inv @ lap
#L_inv - L_inv_jl

Laplacian inverse in Python computed in 7.774 seconds
 --- Loaded Julia packages in 1.605s
 --- Transferred sparse adjacency matrix to Julia in 1.088s
 --- Computed approximate laplacian inverse in Julia in 5.702s


In [4]:
task2 = NodeSelectionTaskFast(adj, probs, true_node)

start_time = time.time()
rscore = task2.resistance_score()
print("Resistance score (normal) %.3f computed in --- %.5f seconds ---" % (rscore, time.time()-start_time))
start_time = time.time()
rscore_fast = task2.resistance_score_fast()
print("Resistance score (fast) %.3f computed in --- %.5f seconds ---" % (rscore_fast, time.time()-start_time))

Resistance score (normal) 0.449 computed in --- 7.92812 seconds ---
 --- Loaded Julia packages in 0.000s
 --- Transferred sparse adjacency matrix to Julia in 0.000s
 --- Computed approximate laplacian inverse in Julia in 3.363s
Resistance score (fast) 0.449 computed in --- 3.49386 seconds ---


--------
From here onwards, it's about a faster implementation of the diffusion score using Krylov subspace fast computation of the matrix exponential times a vector. It is a bit different, because here we don't compute the full matrix exponential, but only its action on a vector, using the fast expm_multiply function from scipy.

In [5]:
task3 = NodeSelectionTaskFast(adj, probs, true_node)
start_time = time.time()
dscore = task3.diffusion_score(kappa2=0.1)
print("Diffusion score (normal) %.6f computed in --- %.5f seconds ---" % (dscore, time.time()-start_time))
start_time = time.time()
dscore_fast = task3.diffusion_score_fast(kappa2=0.1)
print("Diffusion score (fast) %.6f computed in --- %.5f seconds ---" % (dscore_fast, time.time()-start_time))

Diffusion score (normal) -0.000130 computed in --- 3.01093 seconds ---
Diffusion score (fast) -0.000130 computed in --- 0.05902 seconds ---
